# Observational: the joint behavioral RDM on its own

`behav_joint` is the SpAM behavioral RDM with both SHINE cohorts pooled (155 retained participants: 79 shown the original images, 76 the SHINE-processed ones). Like `0a_tree_shape` for the tree, this notebook describes it before notebooks `1_` to `6_` compare it with the tree:

1. **Missing data.** Which pairs were never rated, and are they spread evenly over the tree and over the six image categories?
2. **Noise model.** How consistent is each participant with themselves (test-retest), and how high can any model's agreement with these ratings get (noise ceilings, Nili et al., 2014)?
3. **External resources.** Do the placements differ from random placement, and do they agree with independent similarity ratings of the same images (Mur et al., 2013)?
4. **Decomposition.** How much of the RDM do semantic (KM tree, WordNet) and sensory (CLIP, pixels) predictors explain together and uniquely?

Pixels and CLIP differ between the two image variants, so wherever they enter here they are the mean of their pre- and post-SHINE RDMs (the pooled cohorts saw one variant each, about half and half).

**Run from this directory** with the RDMs built. The participant bootstraps and the noise-ceiling folds are cached under `analysis/results/semantic_hierarchy/`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/s2_semantic_hierarchy/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

import plotly.io as pio

pio.renderers.default = "notebook_connected"  # figures load plotly.js once instead of embedding it

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, wasserstein_distance

from analysis.s0_build_rdms.common import subset_condensed
from analysis.s1_pre_vs_post_shine import decomposition as dec
from analysis.s1_pre_vs_post_shine import ground_truth as gt
from analysis.s1_pre_vs_post_shine.data_quality import (
    pairwise_distance_values, subject_ks_distance, subject_reliability, subject_wasserstein_distance,
)
from analysis.s1_pre_vs_post_shine.reference_agreement import bootstrap_combined_fit_multi
from analysis.s2_semantic_hierarchy import common as c
from analysis.s2_semantic_hierarchy import joint_behavior as jb
from analysis.utils.null_distances import simulate as simulate_null_distances
from analysis.utils.stats import label_shuffle_test, percentile_ci

PARTS = c.tree_parts()
RDMS = c.load_rdm_set([c.REFERENCE, c.BEHAV])
BEHAV, KM = RDMS[c.BEHAV], RDMS[c.REFERENCE]
TRIALS, N_CONDS = jb.joint_trials()
DF_TRIALS, IDS = jb.joint_participants()
REBUILT, COUNTS = jb.combine(TRIALS, list(TRIALS), N_CONDS)
CATEGORIES = jb.image_categories()
same = np.allclose(np.nan_to_num(REBUILT, nan=-1), np.nan_to_num(BEHAV, nan=-1))
print(f"{len(TRIALS)} participants, {len(BEHAV):,} pairs; rebuilt RDM identical to the stored D_behav_joint_simple: {same}")
print("images per category:", pd.Series(CATEGORIES).value_counts().reindex(list(jb.CATEGORY_ORDER)).to_dict())

155 participants, 262,450 pairs; rebuilt RDM identical to the stored D_behav_joint_simple: True
images per category: {'HF': 102, 'HB': 62, 'AF': 53, 'AB': 81, 'NO': 160, 'HO': 267}


## 1. Missing data

**Question.** SpAM trials show about 20 images each, so some pairs are never placed together. Are the unobserved pairs spread evenly over the tree (pooled LCA bins and branch-depth strata, `0a_tree_shape`) and over the six image categories (human face HF, human body HB, animal face AF, animal body AB, natural object NO, handmade object HO)? **Why.** If they clustered somewhere, every raw behavioral estimate would weight that part of the image set less, and an MDS embedding, which fills exactly those pairs, would change the weighting.

In [2]:
COVERAGE = jb.coverage(COUNTS, PARTS, CATEGORIES)
jb.fig_coverage(COVERAGE).show()
jb.fig_category_coverage(jb.category_matrix(COVERAGE), title="Fraction of pairs observed, by category pair").show()
jb.fig_category_coverage(jb.category_matrix(COVERAGE, "ratings_per_observed_pair"),
                         title="Ratings per observed pair, by category pair", fmt=".2f").show()
print(COVERAGE[COVERAGE.index.get_level_values("grouping") != "category pair"].round(3).to_string())
by_pair = COVERAGE.loc["category pair"]
print(f"\ncategory pairs: fraction observed {by_pair['frac_observed'].min():.3f} to {by_pair['frac_observed'].max():.3f}; "
      f"ratings per observed pair {by_pair['ratings_per_observed_pair'].min():.2f} to {by_pair['ratings_per_observed_pair'].max():.2f}")

                  n_pairs  frac_observed  ratings_per_observed_pair
grouping group                                                     
overall  all       262450          0.869                      2.842
LCA bin  0         127246          0.868                      2.847
         1          64696          0.869                      2.853
         2          49272          0.872                      2.823
         3          16130          0.870                      2.823
         >=4         5106          0.865                      2.812
stratum  3x3        56280          0.872                      2.826
         3x>=4     130704          0.868                      2.844
         >=4x>=4    75466          0.867                      2.849

category pairs: fraction observed 0.860 to 0.876; ratings per observed pair 2.77 to 2.91


**Reading.** The missing ratings are spread evenly. 86.9% of the pairs are observed overall, and every pooled LCA bin (86.5% to 87.2%), branch-depth stratum (86.7% to 87.2%) and category pair (86.0% to 87.6%) is within about one percentage point of that. Observed pairs carry 2.8 ratings on average everywhere (2.81 to 2.85 by LCA bin, 2.77 to 2.91 by category pair). So no level of the tree and no category is under-sampled: raw estimates weight them evenly, and an embedding, which fills the missing 13%, does not reweight them.

## 2. Noise model

### 2.1 Test-retest reliability within participants

**Question.** How consistently does a participant arrange the same images when a trial is repeated verbatim (four repeats per participant)? **Why.** It is the noise floor of a single participant's ratings, independent of anyone else. The value per participant is the mean Spearman r between each repeat and its original trial (precomputed by the parser, as in s1 `2_data_quality`).

In [3]:
RELIABILITY = subject_reliability(DF_TRIALS, IDS)
jb.fig_distribution(RELIABILITY, x_title="within-participant test-retest Spearman r",
                    title="Test-retest reliability per participant").show()
q = RELIABILITY.quantile([0.25, 0.5, 0.75])
print(f"n = {len(RELIABILITY)}; median {q[0.5]:.3f} (IQR {q[0.25]:.3f} to {q[0.75]:.3f}); mean {RELIABILITY.mean():.3f}; "
      f"share below 0: {(RELIABILITY < 0).mean():.1%}")

n = 155; median 0.289 (IQR 0.227 to 0.399); mean 0.316; share below 0: 0.0%


**Reading.** Participants are moderately consistent with themselves: the median test-retest Spearman r is 0.289 (IQR 0.227 to 0.399; mean 0.316), and no participant is below 0. This matches the two cohorts in s1 (median 0.276 pre, 0.291 post). A single trial's arrangement therefore carries a real but noisy signal, which is why the group RDM pools 155 participants.

### 2.2 Noise ceiling for a participant's fit (Nili et al., 2014)

**Question.** How well can any model fit a *single participant's* ratings, given how much participants disagree? Nili et al. bound the mean per-participant model fit from below by each participant's agreement with the average of everyone else (leave one out) and from above by their agreement with the average of everyone *including* themselves. Each participant's fit to the KM tree is shown against both bounds, in Kendall tau-a (primary, as KM has nine distinct values) and Spearman.

**A caveat specific to SpAM.** Nili's upper bound assumes each participant contributes about 1/N of every entry of the group average, which holds when everyone rates every pair. Here a pair is rated only a few times, so a participant's own ratings are a large part of the group entry on every pair they rated. `self_share` reports that part directly; the larger it is, the more the upper bound is inflated by participants correlating with themselves.

In [4]:
NILI = jb.nili_bounds(TRIALS, KM, N_CONDS)
jb.fig_nili(NILI, "kendall_a").show()
print(jb.nili_summary(NILI).round(3).to_string())
print(f"\nself_share: median {NILI['self_share'].median():.3f} (IQR {NILI['self_share'].quantile(0.25):.3f} to "
      f"{NILI['self_share'].quantile(0.75):.3f}); 1/N would be {1 / len(NILI):.4f}")
above = (NILI['model_kendall_a'] > NILI['lower_kendall_a']).mean()
print(f"participants whose KM fit exceeds their own lower bound (tau-a): {above:.1%}")

                               mean     sd    n
statistic     quantity                         
Kendall tau-a model fit (KM)  0.187  0.073  155
              lower bound     0.146  0.040  155
              upper bound     0.513  0.031  155
Spearman rho  model fit (KM)  0.285  0.109  155
              lower bound     0.218  0.058  155
              upper bound     0.695  0.037  155

self_share: median 0.430 (IQR 0.428 to 0.433); 1/N would be 0.0065
participants whose KM fit exceeds their own lower bound (tau-a): 87.1%


**Reading.** The KM tree fits a single participant's ratings at a mean tau-a of 0.187 (Spearman 0.285), above the lower bound (0.146; Spearman 0.218) for 87.1% of the participants. That is, the tree predicts a participant's arrangements better than the average of the other 154 participants does, as in s1 (0.186 vs 0.131 pre). The reason is noise in the "others" side: on a participant's pairs, the other participants' average rests on only about 1.6 ratings per pair, while the tree is noise-free.

The upper bound (tau-a 0.513, Spearman 0.695) is not informative here. A participant's own ratings are a median 43.0% of each group entry they rated (IQR 42.8% to 43.3%), against 0.6% (1/N) if everyone rated every pair, so the upper bound mostly measures how well participants correlate with themselves. With this inflation the Nili range is far too wide to say whether the tree is the "true" model; the lower bound is the meaningful one, and the tree clears it.

### 2.3 Noise ceiling for the group RDM

**Question.** Notebooks 1 and 2 compare the *group* RDM with the tree, which needs the reliability of the group RDM itself, not of single participants. **How.** Two disjoint random groups of n participants are drawn (n = 10, 20, 38, 77; 20 draws each) and their combined RDMs are rank-correlated: the empirical reliability of a group RDM of size n. The earlier estimate, the split-half (n = 77 or 78, 100 splits) projected to the full sample with the Spearman-Brown formula, is shown for comparison. It assumes that doubling the group behaves like doubling the test length, which overstates here because extra participants mostly add newly covered pairs rather than more ratings per pair. The full-sample ceiling on a noise-free model's Spearman with the RDM is therefore bracketed: at least sqrt of the 77-participant reliability, at most sqrt of the Spearman-Brown value.

In [5]:
CURVE = jb.group_size_curve(TRIALS, N_CONDS)
RAW = jb.raw_ceiling(TRIALS, N_CONDS)
BRACKET = jb.ceiling_bracket(CURVE, RAW)
jb.fig_group_size(CURVE, RAW, len(TRIALS)).show()
print(CURVE.groupby("group_size")[["spearman", "kendall_a"]].agg(["mean", "std"]).round(3).to_string())
print(f"\nsplit-half Spearman {RAW['spearman'].mean():.3f} -> Spearman-Brown {BRACKET['r_sb']:.3f}")
print(f"ceiling on a noise-free model's Spearman with the full RDM: between {BRACKET['low']:.3f} "
      f"(sqrt of the {BRACKET['group_size']}-participant reliability) and {BRACKET['high']:.3f} (sqrt of Spearman-Brown)")
km_rho = c.rank_agreement(BEHAV, KM)["spearman"]
print(f"for scale: the raw RDM's Spearman with KM is {km_rho:.3f}, i.e. {km_rho / BRACKET['high']:.0%} to {km_rho / BRACKET['low']:.0%} of the ceiling")

           spearman        kendall_a       
               mean    std      mean    std
group_size                                 
10            0.172  0.027     0.115  0.019
20            0.195  0.018     0.130  0.012
38            0.201  0.011     0.135  0.007
77            0.221  0.003     0.148  0.002

split-half Spearman 0.220 -> Spearman-Brown 0.361
ceiling on a noise-free model's Spearman with the full RDM: between 0.470 (sqrt of the 77-participant reliability) and 0.601 (sqrt of Spearman-Brown)


for scale: the raw RDM's Spearman with KM is 0.351, i.e. 58% to 75% of the ceiling


**Reading.** The group RDM's reliability grows slowly with group size: Spearman 0.172 between two groups of 10, 0.195 for 20, 0.201 for 38 and 0.221 for 77. Doubling from 38 to 77 adds only 0.020, whereas the Spearman-Brown formula would predict 0.335 from 0.201. The same formula applied to the split half (0.220) gives the earlier full-sample value of 0.361, which is therefore very likely too high. The ceiling on a noise-free model's Spearman with the full RDM is bracketed between 0.470 (sqrt of the 77-participant reliability, too low for 155 participants) and 0.601 (sqrt of Spearman-Brown, too high). The raw RDM's Spearman with the KM tree, 0.351, is 58% to 75% of that ceiling, which is the range notebooks 1 and 2 use.

A few combinations (7 in the noise-ceiling and bootstrap computations) did not converge within 1000 iterations of the "simple" aggregation and use its last estimate.

## 3. External resources

### 3.1 Distances against random placement

**Question.** Do participants place images differently from chance? Every pairwise distance of every main trial is compared with the distances produced by placing 20 dots uniformly at random (1000 simulated trials, as in s1 `2_data_quality`). Wasserstein distance is primary (it measures how far the distributions are apart), KS distance secondary (shape only); per participant, each is the mean over that participant's trials.

In [6]:
NULL = simulate_null_distances(num_dots=20, num_trials=1000, seed=42)
MAIN = jb.main_trials(DF_TRIALS, IDS)
DISTANCES = pairwise_distance_values(MAIN)
jb.fig_null_distances(DISTANCES, NULL).show()
w_pooled = wasserstein_distance(DISTANCES, NULL)
ks = ks_2samp(DISTANCES, NULL)
print(f"pooled ({len(DISTANCES):,} distances): Wasserstein {w_pooled:.4f}; KS D {ks.statistic:.4f} (p {ks.pvalue:.1e}, descriptive)")
W = subject_wasserstein_distance(MAIN, NULL)
KS = subject_ks_distance(MAIN, NULL)
print(f"per participant: Wasserstein median {W.median():.4f} (IQR {W.quantile(0.25):.4f} to {W.quantile(0.75):.4f}); "
      f"KS median {KS.median():.4f} (IQR {KS.quantile(0.25):.4f} to {KS.quantile(0.75):.4f})")
print(f"mean distance: participants {np.mean(DISTANCES):.3f}, null {NULL.mean():.3f}; SD participants {np.std(DISTANCES):.3f}, null {NULL.std():.3f}")

pooled (647,900 distances): Wasserstein 0.0275; KS D 0.0544 (p 0.0e+00, descriptive)


per participant: Wasserstein median 0.0410 (IQR 0.0313 to 0.0552); KS median 0.1316 (IQR 0.1067 to 0.1653)
mean distance: participants 0.363, null 0.369; SD participants 0.202, null 0.175


**Reading.** Participants place images measurably differently from random placement: over all 647,900 distances the Wasserstein distance to the null is 0.0275 (KS D 0.054), and per participant the Wasserstein distance has a median of 0.041 (IQR 0.031 to 0.055), as in s1 (pooled 0.027 pre, 0.028 post). The mean distance is almost the same as the null's (0.363 vs 0.369), but the spread is larger (SD 0.202 vs 0.175): participants put more images both close together and far apart than random placement would, that is, they form groups.

### 3.2 Agreement with the Mur et al. (2013) ratings

**Question.** Does the joint RDM agree with independent similarity judgements of the same images? 42 of our images also appear in the 92-image set of Kriegeskorte et al. (2008b), which 16 raters arranged in Mur et al. (2013); the overlap was matched by image content in s1 `5_ground_truth`. Agreement is the Kendall tau-a (primary) and Spearman between our RDM and the raters' mean RDM over the pairs we observed, with a participant-bootstrap CI (500 draws), an image-shuffle permutation p (5000), each rater's own fit, and the raters' leave-one-out agreement (their own noise ceiling).

In [7]:
OVERLAP = gt.load_overlap()
ROWS = OVERLAP["manifest_row"].to_numpy()
RATERS = gt.external_targets(OVERLAP)["behavioral_mur2013"]
MUR = gt.mean_rdm(RATERS)
ours = subset_condensed(BEHAV, ROWS)
perm = label_shuffle_test(ours, MUR, n_perm=5000, seed=jb.SEED, cache_path=c.cache_path("mur_permutation_joint_n5000.npz"))
boot = bootstrap_combined_fit_multi(TRIALS, {"mur2013": MUR}, N_CONDS, n_boot=500, seed=jb.SEED, subset=ROWS,
                                    cache_path=c.cache_path("mur_bootstrap_joint_n500.pkl"))
lo, hi = percentile_ci(boot["kendall_a"])
per_rater = gt.per_external_fit(ours, RATERS)
print(f"{len(OVERLAP)} shared images, {len(MUR)} pairs, {int((~np.isnan(ours)).sum())} observed by us")
print(f"tau-a {perm.observed['kendall_a']:.3f} [{lo:.3f}, {hi:.3f}] (bootstrap mean {boot['kendall_a'].mean():.3f}), "
      f"Spearman {perm.observed['spearman']:.3f}; "
      f"permutation p (tau-a) {perm.p_value['kendall_a']:.4f}")
print(f"mean fit to each rater (tau-a): {per_rater.mean():.3f}; raters' leave-one-out agreement: {gt.external_leave_one_out(RATERS):.3f}")

42 shared images, 861 pairs, 736 observed by us
tau-a 0.280 [0.232, 0.299] (bootstrap mean 0.264), Spearman 0.418; permutation p (tau-a) 0.0002
mean fit to each rater (tau-a): 0.239; raters' leave-one-out agreement: 0.355


**Reading.** On the 42 shared images (736 of the 861 pairs observed by us), the joint RDM agrees with the Mur et al. (2013) raters' mean at tau-a 0.280 [0.232, 0.299] (Spearman 0.418), far above the image-shuffle null (p = 0.0002, the floor for 5000 permutations). That is higher than either cohort alone in s1 (0.258 pre, 0.270 post), as expected from pooling more participants. Our RDM agrees with individual Mur raters at 0.239 on average, below the raters' agreement with each other (leave-one-out 0.355), so it is closer to the raters' consensus than to any single rater. The external data have no human faces, so this concerns objects, animals and body parts only.

As in section 4, the participant-bootstrap draws sit below the full-sample estimate (bootstrap mean 0.264 vs 0.280), so the CI is shifted down.

## 4. Decomposition into semantic and sensory parts

**Question.** How much of the joint RDM do semantic and sensory predictors explain, and how is it shared between them? The model is the pre-registered H3 decomposition of s1 `6_decomposition`, `D_perc = a*D_sem + b*D_CLIP + g*D_pix + e`, fit by non-negative least squares on z-scored RDMs over the observed pairs, so the betas are standardized. Three nested models: KM + pixels, KM + CLIP + pixels, and KM + WordNet + CLIP + pixels. Hierarchical delta R^2 enters pixels first, then CLIP, then the semantic predictors. CIs are a participant bootstrap (1000 draws, each re-combining resampled participants). Pixels and CLIP are the mean of their pre- and post-SHINE RDMs.

In [8]:
PREDICTORS = jb.joint_predictors()
OBSERVED = dec.fit_models(BEHAV, PREDICTORS, models=jb.JOINT_MODELS)
BOOTS = jb.bootstrap_fits(TRIALS, PREDICTORS, N_CONDS, cache_path=c.cache_path("decomposition_joint_n1000.pkl"))
BETAS = jb.fit_estimates(OBSERVED, BOOTS, "beta")
jb.fig_betas(BETAS).show()
for stat, title in [("beta", "standardized NNLS beta"), ("partial_r", "partial correlation"), ("delta_r2", "hierarchical delta R^2")]:
    est = jb.fit_estimates(OBSERVED, BOOTS, stat)
    est["value"] = est.apply(lambda r: f"{r.estimate:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}] (boot mean {r.boot_mean:.3f})", axis=1)
    print(f"\n{title} (95% CI)")
    print(est.pivot(index="term", columns="model", values="value").reindex(list(dec.TERMS)).fillna("").to_string())
r2 = jb.fit_estimates(OBSERVED, BOOTS, "r2").drop_duplicates("model")
print("\nmodel R^2 (NNLS):", {r.model: f"{r.estimate:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}] (boot mean {r.boot_mean:.3f})" for r in r2.itertuples()})


standardized NNLS beta (95% CI)
model                                       km                                 km+clip                              km+wn+clip
term                                                                                                                          
sem_km  0.385 [0.331, 0.367] (boot mean 0.350)  0.306 [0.260, 0.295] (boot mean 0.278)  0.270 [0.227, 0.261] (boot mean 0.245)
sem_wn                                                                                  0.111 [0.090, 0.111] (boot mean 0.101)
clip                                            0.209 [0.180, 0.198] (boot mean 0.190)  0.204 [0.176, 0.194] (boot mean 0.186)
pix     0.094 [0.079, 0.092] (boot mean 0.085)  0.051 [0.040, 0.052] (boot mean 0.046)  0.055 [0.043, 0.055] (boot mean 0.049)

partial correlation (95% CI)
model                                       km                                 km+clip                              km+wn+clip
term                                            


model R^2 (NNLS): {'km': '0.164 [0.123, 0.148] (boot mean 0.136)', 'km+clip': '0.199 [0.151, 0.178] (boot mean 0.165)', 'km+wn+clip': '0.210 [0.160, 0.187] (boot mean 0.174)'}


**Reading.** The KM tree is the strongest predictor in every model: its standardized beta is 0.385 with pixels only, 0.306 once CLIP is added and 0.270 once WordNet is added too. CLIP carries a substantial separate part (beta 0.209 and 0.204), WordNet a small one (0.111), and pixels little (0.051 to 0.094). The models explain 16.4%, 19.9% and 21.0% of the rank variance. In the hierarchical order (pixels, then CLIP, then semantics), pixels add 0.018, CLIP 0.103, the KM tree a further 0.078 and WordNet 0.011: after both sensory predictors are in, the tree still explains about three quarters as much as CLIP added, so the semantic structure in behavior is not just CLIP's.

**On the CIs.** Every participant-bootstrap distribution sits below its full-sample estimate (for example the KM beta in the full model: estimate 0.270, bootstrap mean 0.245; R^2 0.210 vs 0.174). A resample of 155 participants drawn with replacement contains only about 63% distinct participants, and on these sparse RDMs that behaves like a smaller, noisier group, which attenuates every fit. The percentile CIs therefore describe a smaller sample and understate the full-sample values; the ordering of the predictors (KM, then CLIP, then WordNet, then pixels) is the same in all 1000 draws of every model.

## 5. Takeaways

1. **Missing ratings are spread evenly.** 86.9% of pairs are observed, within about one percentage point in every LCA bin, branch-depth stratum and category pair, at 2.8 ratings per observed pair.
2. **Participants are moderately consistent with themselves:** median test-retest Spearman r 0.289 (IQR 0.227 to 0.399), none below 0.
3. **The tree fits single participants better than the other participants do:** mean tau-a 0.187 against a Nili lower bound of 0.146, above it for 87.1% of participants. Nili's upper bound (0.513) is not usable here, because each participant is about 43% of every group entry they rated.
4. **The group RDM's reliability grows slowly with group size** (Spearman 0.172 at 10, 0.221 at 77 participants per group), so the Spearman-Brown projection (0.361) overstates the full sample. The ceiling on a noise-free model's Spearman with the full RDM lies between 0.470 and 0.601; the raw RDM's 0.351 with the tree is 58% to 75% of it.
5. **Placements are not random:** Wasserstein distance to random placement 0.0275 pooled (per participant median 0.041), with more close and far placements than chance (SD 0.202 vs 0.175).
6. **The joint RDM agrees with independent ratings:** tau-a 0.280 [0.232, 0.299] with the Mur et al. (2013) raters on 42 shared images, above either cohort alone.
7. **Semantics explains behavior beyond the sensory predictors.** With pixels and CLIP in the model, the KM tree still adds 0.078 of R^2 (about three quarters of CLIP's 0.103), WordNet 0.011; full-model R^2 0.210, KM beta 0.270, CLIP 0.204, WordNet 0.111, pixels 0.055.

**Limitations.** Participant-bootstrap CIs (sections 3.2 and 4) are shifted below the full-sample estimates, because a resample has only about 63% distinct participants; they understate the full-sample values. The ceiling bracket rests on group sizes up to 77; the full sample's reliability is not measured directly. Pixels and CLIP are the mean of the two image variants, while each participant saw only one. The Mur comparison has no human faces.